In previouos we created a Simple Q&A AI application, Now we will create it with **Langchain** for more convienent

In [1]:
import os

In [ ]:
openai_api_keys = os.environ['OPENAI_API_KEYS'] #Load the openai api keys or use the alternative open router platform to get free api keys

In [3]:
llm_model = "gpt-3.5-turbo" #Load the LLM Model or any openai model that is available

In [ ]:
from langchain.chat_models import ChatOpenAI

In [ ]:
chat = ChatOpenAI(temperature=0.0, model=llm_model)

We can import prompt templates for the output in your specific format from AI

In [ ]:
from langchain.prompts import ChatPromptTemplate

In [ ]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [ ]:
prompt_template = ChatPromptTemplate.from_template(template_string)

In [5]:
customer_style = """American English \ in a calm and respectful tone """

In [6]:
customer_email = """Arrr, I be fuming that me blender lid \ flew off and splattered me kitchen walls \ with smoothie! And to make matters worse, \
the warranty don't cover the cost of \ cleaning up me kitchen. I need yer help \ right now, matey!  """

In [ ]:
customer_messages = prompt_template.format_messages(style=customer_style,text=customer_email)

In [ ]:
# Call the LLM to translate to the style of the customer message
customer_response = chat(customer_messages)

In [ ]:
print(customer_response.content)

**Output** : Oh man, I'm really frustrated that my blender lid flew off and made a mess of my kitchen walls with smoothie! And on top of that, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now, buddy!

**Let's say the output should be in the (key, value) dictionary format**

In [ ]:
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

In [ ]:
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [ ]:
format_instructions = output_parser.get_format_instructions()

In [ ]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)

In [ ]:
response = chat(messages)

**Output** : 
{
	"gift": true,
	"delivery_days": 2,
	"price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}